# TOKATÇI — Value model (Kaggle)

Amaç: **state → maçı kazanma olasılığı**.

- BC action label’ı kullanılmaz.
- Sadece public state feature’ları kullanılır.
- Episode-level split korunur.
- B1/B2/B2C aynı state feature’larını kullanır; bu notebook **ortak value modeli** üretir.
- Search tarafında leaf değerlendirmesi için kullanılacak.


In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch import nn

INPUT = Path('/kaggle/input')
DATA = next(INPUT.rglob('dataset_tokatci_b1.npz'), None)
if DATA is None:
    raise FileNotFoundError('dataset_tokatci_b1.npz bulunamadı; tokatci_data input ekle.')
OUT = Path('/kaggle/working/model_tokatci_value.npz')
print('DATA', DATA)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


In [ ]:
d = np.load(DATA)
state = d['state'].astype(np.float32)          # [N, 50]
y = d['game_win'].astype(np.float32)           # [N]
w = d['sample_weight'].astype(np.float32)      # [N]
split = d['split'].astype(np.int8)
ep = d['episode_id']

train_idx = np.nonzero(split == 0)[0]
val_idx = np.nonzero(split == 1)[0]
print('rows', len(state), 'train', len(train_idx), 'val', len(val_idx), 'episodes', len(np.unique(ep)))
print('win rate', float(y.mean()))


In [ ]:
# Normalizer: yalnızca train state üzerinden.
mean = state[train_idx].mean(axis=0)
std = state[train_idx].std(axis=0)
weak = std < 1e-4
mean[weak] = 0.0
std[weak] = 1.0
np.savez('/kaggle/working/value_normalizer.npz', state_mean=mean.astype(np.float32), state_std=std.astype(np.float32))
print('state dim', state.shape[1])


In [ ]:
class ValueNet(nn.Module):
    def __init__(self, dim=50, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = ValueNet(state.shape[1]).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss(reduction='none')
print('device', device, 'params', sum(p.numel() for p in net.parameters()))


In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate(net, idx, chunk=8192):
    net.eval(); logits=[]; labels=[]
    with torch.no_grad():
        for start in range(0,len(idx),chunk):
            ii=idx[start:start+chunk]
            xb=torch.from_numpy((state[ii]-mean)/std).to(device, non_blocking=True)
            logits.append(net(xb).cpu().numpy())
            labels.append(y[ii])
    p=np.concatenate(logits); yy=np.concatenate(labels)
    prob=1/(1+np.exp(-p))
    bce=float(np.mean(np.maximum(p,0)-p*yy+np.log1p(np.exp(-np.abs(p)))))
    acc=float(((prob>=0.5)==(yy>=0.5)).mean())
    auc=float(roc_auc_score(yy,prob))
    brier=float(np.mean((prob-yy)**2))
    return bce,acc,auc,brier

best=(-1,None,0)
patience=8; bad=0; batch_size=1024
for epoch in range(1,31):
    net.train()
    order=train_idx.copy(); np.random.shuffle(order)
    for start in range(0,len(order),batch_size):
        ii=order[start:start+batch_size]
        xb=torch.from_numpy((state[ii]-mean)/std).to(device, non_blocking=True)
        yb=torch.from_numpy(y[ii]).to(device, non_blocking=True)
        wb=torch.from_numpy(w[ii]).to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        logits=net(xb)
        loss=(loss_fn(logits,yb)*wb).sum()/wb.sum().clamp_min(1e-8)
        loss.backward(); opt.step()
    val=evaluate(net,val_idx)
    print(f'epoch {epoch:2d} val bce={val[0]:.4f} acc={val[1]:.4f} auc={val[2]:.4f} brier={val[3]:.4f}')
    score=val[2] - 0.25*val[0]  # AUC ana, BCE ceza
    if score>best[0]+1e-9:
        best=(score,{k:v.detach().cpu().clone() for k,v in net.state_dict().items()},epoch)
        bad=0
    else:
        bad+=1
        if bad>=patience:
            print('early stop'); break

net.load_state_dict(best[1])
val=evaluate(net,val_idx)
print('BEST epoch',best[2],'val',val)


In [ ]:
# Save NumPy weights for search/inference.
# CUDA tensor -> CPU -> numpy.  Direct .numpy() fails on CUDA tensors.
arrays = {k: v.detach().cpu().numpy() for k, v in net.state_dict().items()}
arrays['state_mean'] = mean.astype(np.float32)
arrays['state_std'] = std.astype(np.float32)
np.savez_compressed(OUT, **arrays)
print('wrote', OUT, OUT.stat().st_size / 1024, 'KB')


## Search tarafında kullanım

```text
value = sigmoid(ValueNet(normalize(state)))
```

Search leaf’inde bu değer kullanılacak. BC prior sadece dallanma azaltmak için, düşük ağırlıkla.
